# Computational Set: Data Analysis C — Weighted Linear Regression

### Activation Energy from Reaction Kinetics

**Learning objectives**

- Understand why data points with larger uncertainties should carry **less weight** in a fit.
- Perform a **weighted** linear regression and extract parameters with uncertainties.
- Display data with **error bars** and obtain a physical quantity (activation energy) from a slope.

---

#### Background

In Exercises A and B we assumed every $y$ value was equally reliable. Often some measurements
are more uncertain than others and should count for less. We handle this with **weights**: a
point with uncertainty $\sigma_i$ gets weight $w_i = 1/\sigma_i^2$.

Here we analyze the decomposition of the benzenediazonium ion. The temperature dependence of the
rate constant $k$ follows the Arrhenius equation, which is linear in $1/T$ when written
logarithmically:

$$
\ln k = -\frac{E_a}{R}\,\frac{1}{T} + \ln A. \tag{6}
$$

So a plot of $\ln k$ vs $1/T$ has slope $-E_a/R$. Because each $k$ has its own uncertainty, the
uncertainty in $\ln k$ is

$$
\Delta(\ln k)_i = \left|\frac{\Delta k}{k_i}\right|, \tag{7}
$$

and we weight each point by $1/[\Delta(\ln k)_i]^2$.


> **New syntax in this set.**
> - `np.abs(x)` gives the absolute value, element by element.
> - `ax.errorbar(x, y, yerr=..., fmt='o')` draws points **with vertical error bars**.
> - `np.polyfit` accepts a **weight** argument, `w=...`. For Gaussian uncertainties NumPy wants
>   `w = 1/σ` (the *reciprocal of the standard deviation*, **not** $1/\sigma^2$) — here that is
>   `w = 1/Δ(ln k)`. With these weights, `polyfit(..., cov=True)` reproduces the weighted-regression
>   standard errors you got from the Excel Regression tool.
>
> Fuller refresher: ESCIP [“What is Python?”](https://escip.io/notebooks/python/python-basics-fixed.html).

# The data

For each temperature `T_C` (°C) we have the measured rate constant `k` (s⁻¹) and its standard
deviation `sk` (s⁻¹) from replicate runs.

# 💾 Run the cell below to load the data — **you don't need to edit it.**

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import FileUpload, Dropdown, VBox
from IPython.display import display
import io
import pandas as pd

uploader = FileUpload(accept='.csv', multiple=False, description='Upload CSV')
display(VBox([uploader]))

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# 📥 Run the cell below to load your data - **you don't need to edit it.**

In [4]:
# Run this cell AFTER you've uploaded your file above.

if len(uploader.value) == 0:
    raise ValueError("No file uploaded yet — click 'Upload CSV' above, choose your file, then re-run this cell.")

# ipywidgets FileUpload.value is a tuple of dicts (v8+) or a dict keyed by filename (v7)
uploaded = uploader.value[0] if isinstance(uploader.value, tuple) else list(uploader.value.values())[0]
raw = pd.read_csv(io.BytesIO(bytes(uploaded['content'])), header=None)

# --- First three columns: T(deg C), k(inverse seconds), delta k (inverse seconds) ---
data = raw.iloc[1:, 0:3].reset_index(drop=True).astype(float)
data.columns = raw.iloc[0, 0:3].tolist()

T  = data.iloc[:, 0].values
k  = data.iloc[:, 1].values
dk = data.iloc[:, 2].values



print(f"{len(T)} data points loaded from file.")
print(f"Temperature = {T} °C,  k = {k} in s^-1, uncertainty in k {dk} in s^-1")


6 data points loaded from file.
Temperature = [24.4 30.  34.7 40.2 45.1 50.3] °C,  k = [7.30e-05 1.39e-04 3.53e-04 7.17e-04 1.60e-03 3.34e-03] in s^-1, uncertainty in k [3.0e-05 5.0e-05 7.0e-05 1.2e-04 1.7e-04 1.5e-04] in s^-1


## Step 1 — Build the derived columns

To linearize Arrhenius (Eq. 6) you need, with $T$ in **kelvin**:

- `invT` — the inverse absolute temperature (the $x$ data),
- `lnk`  — the natural log of the rate constant (the $y$ data),
- `dlnk` — the uncertainty in $\ln k$ from Eq. (7): the magnitude of (uncertainty in $k$) divided
  by ($k$ itself).

👉 **Your task:** complete the four expressions.

In [ ]:
# absolute temperature in kelvin
T_K =                    # <-- Celsius value + 273.15

# x data: inverse absolute temperature
invT =                   # <-- replace with your expression

# y data: natural log of the rate constant
lnk =                    # <-- replace with your expression

# uncertainty in ln k (Eq. 7): magnitude of (sk / k)
dlnk =                   # <-- use np.abs(...)

print("1/T :", np.round(invT, 6))
print("ln k:", np.round(lnk, 3))
print("Δlnk:", np.round(dlnk, 4))

## Step 2 — Plot the data with error bars

Plot $\ln k$ (vertical) against $1/T$ (horizontal), with a vertical error bar on each point equal
to `dlnk`. Larger bars mark the less-trustworthy points.

👉 **Your task:** supply the `ax.errorbar(...)` call.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# Plot invT (x) vs lnk (y) as points with vertical error bars given by dlnk.
ax.errorbar(                       )   # <-- pass x, y, yerr=dlnk, fmt='o', mfc='none', capsize=3

ax.set_xlabel(r'$1/T$ (K$^{-1}$)')
ax.set_ylabel(r'$\ln k$')
ax.tick_params(direction='in')
plt.show()

## Step 3 — Weighted linear regression

Fit a straight line, but pass the **weights** so uncertain points count for less. The weight for
each point is the reciprocal of its $\ln k$ uncertainty.

👉 **Your task:** build the weights and run the weighted fit.

In [ ]:
# weights: reciprocal of the ln k uncertainties
w =                      # <-- 1 / dlnk

# weighted degree-1 fit with covariance
coeffs, cov =            # <-- np.polyfit(invT, lnk, 1, w=w, cov=True)

slope, intercept = coeffs
S_slope, S_intercept = np.sqrt(np.diag(cov))

print(f"slope     = {slope:.1f}  (S = {S_slope:.1f})  K")
print(f"intercept = {intercept:.2f}  (S = {S_intercept:.2f})  (= ln A)")

## Step 4 — Activation energy from the slope

From Eq. (6) the slope equals $-E_a/R$, so $E_a = -\text{slope}\times R$. Because this is a single
multiplication, the uncertainty scales the same way: $\Delta E_a = S_\text{slope}\times R$.

👉 **Your task:** compute the activation energy and its uncertainty (in J/mol).

In [ ]:
# activation energy and its uncertainty, in J/mol
Ea =                     # <-- -slope * R_gas
S_Ea =                   # <-- S_slope * R_gas

print(f"Ea = {Ea/1000:.1f} ± {S_Ea/1000:.1f} kJ/mol")

## Step 5 — Overlay the fit and report

Plot the weighted regression line over the data, then report the activation energy with correct
significant figures using the `report` helper.

👉 **Your task:** build the predicted line and call `report` for $E_a$ (in kJ/mol).

In [ ]:
from math import log10, floor

# Format 'value +/- uncertainty unit' following the lab's two sig-fig rules.
def report(value, unc, unit=""):
    if unc == 0:
        return f"{value} {unit}"
    exp = floor(log10(abs(unc)))
    lead = int(abs(unc) / 10**exp)
    sig = 2 if lead in (1, 2) else 1
    dp = -(exp - (sig - 1))
    if dp >= 0:
        return f"{value:.{dp}f} ± {unc:.{dp}f} {unit}"
    f = 10**(-dp)
    return f"{round(value/f)*f:g} ± {round(unc/f)*f:g} {unit}"

# predicted ln k from the fitted line:  slope * invT + intercept
lnk_calc =               # <-- replace with your expression

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(invT, lnk, yerr=dlnk, fmt='o', mfc='none', capsize=3, label='data')
ax.plot(invT, lnk_calc, '-', lw=1, label='weighted fit')
ax.set_xlabel(r'$1/T$ (K$^{-1}$)')
ax.set_ylabel(r'$\ln k$')
ax.tick_params(direction='in'); ax.legend()
plt.show()

print("Ea =", report(             ))   # <-- pass Ea/1000, S_Ea/1000, "kJ/mol"


## Discussion

Answer briefly below (as Markdown):

1. Which points have the largest error bars, and how does weighting change their influence on the slope?
2. Try an **unweighted** fit (`np.polyfit(invT, lnk, 1)`) and compare the slope/Eₐ. How different is it, and why?
3. Which measurement limits the precision of $E_a$ most? How would you improve it experimentally?

*Your answers here.*